# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Fetch record sets and display details with their @id

record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Available record sets and their fields:")
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            for field in rs['field']:
                print(f"  Field @id: {field['@id']} - {field.get('name', '')}")
        else:
            print("  No fields defined for this record set.")

# If no record sets are defined in the metadata as a list, fallback to loading records by inspection
# Otherwise, use the @id for subsequent blocks

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If multiple record sets are available, their IDs are listed below.

In [ ]:
# Identify record set IDs

record_set_ids = []
if record_sets:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])
else:
    # If recordSet not present, try loading records and see available keys
    # Often, datasets define one record set. Let's try loading with inspection.
    sample_records = list(dataset.records())
    if sample_records:
        print("Record keys from sample records:")
        print(sample_records[0].keys())
    else:
        print("No records available to inspect.")

dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record_set '{record_set_id}' with columns:")
        print(df.columns.tolist())
        print(df.head())
else:
    # Use sample_records if no explicit record sets
    df = pd.DataFrame(sample_records)
    dataframes['default'] = df
    print("Loaded default DataFrame with columns:")
    print(df.columns.tolist())
    print(df.head())

# Set up a variable for downstream analysis
main_recordset_id = record_set_ids[0] if record_set_ids else 'default'

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations can include removing outliers, transforming numeric fields, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Choose a numeric field for EDA. As per dataset description, GAD-7, PHQ-9, PSQ scores are likely candidates.
# We'll check which exist in the DataFrame.

numeric_fields = ['GAD-7', 'PHQ-9', 'PSQ']
df = dataframes[main_recordset_id]
available_numeric = [col for col in df.columns if col in numeric_fields]

print(f"Available numeric fields: {available_numeric}")
if available_numeric:
    numeric_field = available_numeric[0]  # e.g., GAD-7
else:
    numeric_field = df.select_dtypes(include='number').columns[0] if df.select_dtypes(include='number').shape[1] > 0 else None

if numeric_field is not None:
    # Using threshold for filtering (e.g., GAD-7 scores above 10 indicate moderate/severe anxiety)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a demographic field if available
    demographic_fields = ['gender', 'Gender', 'level_of_education', 'Level of Education', 'village', 'Village']
    group_field = next((col for col in df.columns if col in demographic_fields), None)
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No demographic group field found for grouping.")
else:
    print("No numeric field found in the record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot histogram of numeric field, and boxplot/grouped bar by demographic if available

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field found previously, plot grouped bar
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field, data=df, ci=None)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich mental health survey data with multiple numeric and demographic fields.
- Using the `mlcroissant` library, we loaded the Croissant schema and accessed records by their `@id`.
- Exploratory analysis highlighted distributions of commonly used mental health scores (e.g., GAD-7).
- Grouped analyses by demographic attributes can inform targeted community interventions.
- Further work could involve deeper statistical modeling or expanding visualizations to trends over time or additional fields.